In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from scipy.stats import wilcoxon
from dataclasses import dataclass
from matplotlib.figure import Figure
from matplotlib.axes import Axes
from definitions import Algorithm, Metric, DataColumn, Alternative, ALGORITHM_METADATA

@dataclass(frozen=True)
class PlotConfig:
    data: pd.DataFrame
    backbone_metric: Metric
    dimensions: tuple[int, int]
    edge_property: DataColumn
    algorithms: Algorithm
    round: int
    comparisons: list[tuple[int, int]]
    wilcoxon_alternative: Alternative

In [ ]:
def handle_canvas(row_count: int, column_count: int) -> tuple[Figure, Axes]:
    figure_height = 5 * row_count
    figure_width = 5 * column_count

    fig, axes = plt.subplots(
        nrows=row_count,
        ncols=column_count,
        figsize=(figure_width, figure_height),
        sharey=True,
        gridspec_kw={"hspace": 0.27}
    )

    return fig, axes


def handle_plot_titles(partition_ranges: np.ndarray, round: int) -> list[str]:
    ranges = np.round(partition_ranges, round).tolist()

    bin_labels: list[str] = []

    head_label = f" ≤ {ranges[1]}"
    bin_labels.append(head_label)

    for i in range(len(ranges) - 3):
        between_label = f"{ranges[i + 1]} - {ranges[i + 2]}"
        bin_labels.append(between_label)

    tail_label = f" ≥ {ranges[len(ranges) - 2]}"
    bin_labels.append(tail_label)
    
    return bin_labels


def handle_p_values(p: float) -> str:
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


def handle_brackets(ax: Axes, x1: float, x2: float, y: float, h: float, text: str) -> None:
    ax.plot(
        [x1, x1, x2, x2], 
        [y, y + h, y + h, y], 
        lw=1.3, 
        c="black"
    )

    ax.text(
        (x1 + x2) / 2, 
        y + h, 
        text, 
        ha="center", 
        va="bottom"
    )


def handle_plot_labels(ax: Axes, plot_subtext: str) -> None:
    ax.set_title(
        plot_subtext,
        fontsize=14   
    )
    ax.tick_params(axis="x", labelsize=12)
    ax.tick_params(axis="y", labelsize=10)

    ax.margins(y=0.15)

    ax.set_yscale("log")


def handle_colors(plots: dict) -> None:
    for median in plots["medians"]:
        median.set_color("black")
        median.set_linewidth(1)

    for patch, algorithm in zip(plots["boxes"], Algorithm):
        metadata = ALGORITHM_METADATA[algorithm]
        color = metadata.color
        
        patch.set_facecolor(color)


def handle_significance(ax: Axes, data_partition: pd.DataFrame, metric: Metric, comparisons: list[int, int], alternative: str):
    algorithm_runtime_columns = [ALGORITHM_METADATA[algorithm].get_runtime_column(metric) for algorithm in list(Algorithm)[:3]]

    y_max = max(data_partition[c].max() for c in algorithm_runtime_columns) 
    base = y_max * 1.5
    factor = 3
    level = 0

    for (a, b) in comparisons:
        stat, p = wilcoxon(
            x=data_partition[algorithm_runtime_columns[a]],
            y=data_partition[algorithm_runtime_columns[b]],
            alternative=alternative
        )

        if p < 0.05:
            new_star = handle_p_values(p)

            y = base * (factor ** level) 
            handle_brackets(
                ax=ax,
                x1=a + 1,
                x2=b + 1,
                y=y,
                h=y*0.02,
                text=new_star
            )
            level += 1


def handle_figure_x_label(fig: Figure, axes: Axes, x_label: str) -> None:
    fig.canvas.draw()
    top_y = max(ax.get_position().y1 for ax in axes.flat)
    offset = mtransforms.ScaledTranslation(
        0, 40 / 72, fig.dpi_scale_trans  # 8 points upward
    )
    
    fig.text(
        0.5, top_y,
        x_label,
        ha="center",
        va="bottom",
        fontsize=16,
        transform=fig.transFigure + offset
    )


def handle_figure_y_label(fig: Figure, axes: Axes, y_label: str) -> None:
    fig.canvas.draw()

    left_x = min(ax.get_position().x0 for ax in axes.flat)

    offset = mtransforms.ScaledTranslation(
        -60 / 72, 0, fig.dpi_scale_trans
    )

    fig.text(
        left_x,
        0.5,
        y_label,
        ha="center",
        va="center",
        rotation=90,
        fontsize=16,
        transform=fig.transFigure + offset,
    )


def handle_p_box(fig: Figure, axes: Axes) -> None:
    fig.canvas.draw()

    right = max(ax.get_position().x1 for ax in axes.flat)
    top = max(ax.get_position().y1 for ax in axes.flat)

    offset = mtransforms.ScaledTranslation(
        18 / 72, -2.5 / 72, fig.dpi_scale_trans
    )

    fig.text(
        right,
        top,
        "*  p < 0.05\n** p < 0.01\n*** p < 0.001",
        ha="left",
        va="top",
        fontsize=12,
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="black"),
        transform=fig.transFigure + offset,
    )


def handle_figure(fig: Figure, axes: Axes, x_label: str, y_label: str) -> None:
    handle_figure_x_label(
        fig=fig,
        axes=axes,
        x_label=x_label
    )

    handle_figure_y_label(
        fig=fig,
        axes=axes,
        y_label=y_label
    )

    handle_p_box(
        fig=fig,
        axes=axes
    )


def handle_plots(ax: Axes, plot: dict, subtext: str, next_partition: pd.DataFrame, config: PlotConfig):
    handle_plot_labels(
        ax=ax, 
        plot_subtext=subtext
    )

    handle_significance(
        ax=ax,
        data_partition=next_partition,
        metric=config.backbone_metric,
        comparisons=config.comparisons,
        alternative=config.wilcoxon_alternative
    )

    handle_colors(plots=plot)


def draw_plot(config: PlotConfig) -> None:
    df = config.data
    rows = config.dimensions[0]
    cols = config.dimensions[1]
    property = config.edge_property

    fig, axes = handle_canvas(
        row_count=rows, 
        column_count=cols,
    )

    x_label = f"{config.backbone_metric.capitalize()} Backbone of {len(df)} Benchmark Networks Evenly Partitioned by {config.edge_property.capitalize()}"
    y_label = f"Runtime (s)"
    handle_figure(
        fig, axes,
        x_label=x_label,
        y_label=y_label
    )

    total_plots = axes.size 
    partition_ids = [i for i in range(total_plots)]
    
    new_column = property + "_bin"
    df[new_column], partitions = pd.qcut(
        df[property],
        q=total_plots,
        labels=partition_ids,
        retbins=True
    )

    titles = handle_plot_titles(
        partition_ranges=partitions, 
        round=config.round
    )

    for ax, partition_id, subtext, in zip(axes.flat, partition_ids, titles):
        next_partition = df[df[new_column] == partition_id]
        
        algorithm_labels: list[str] = []
        data_to_plot: list[pd.DataFrame] = []

        for algorithm in list(Algorithm)[:3]:
            metadata = ALGORITHM_METADATA[algorithm]

            runtime_column = metadata.get_runtime_column(config.backbone_metric)
            runtime_data = next_partition[runtime_column]

            data_to_plot.append(runtime_data)
            algorithm_labels.append(algorithm.capitalize())
        
        plot = ax.boxplot(
            data_to_plot,
            patch_artist=True,
            tick_labels=algorithm_labels
        )

        handle_plots(
            ax, plot,
            subtext=subtext,
            next_partition=next_partition,
            config=config
        )


plot_config = PlotConfig(
    data="",
    backbone_metric=Metric.METRIC,
    dimensions=(2, 4),
    edge_property=DataColumn.DENSITY,
    algorithms=Algorithm,
    round=2, 
    comparisons=[(0, 1), (0, 2), (1, 2)],
    wilcoxon_alternative=Alternative.LESS
)

draw_plot(plot_config)